#### Laplace -- diagonale Laplace-Approximation auf Qwen-LoRA

Nach Daxberger et al. 2021 ("Laplace Redux"), diagonale empirische Fisher-Naeherung. Baut auf `qwen_posterior_utils.py` auf (gleicher `theta_map`, gleiches 128er-Posterior-Subset, gleiche Auswertungspipeline wie MILE/MFVI). Kein Trainingsloop noetig -- ein einzelner Durchlauf ueber die Posterior-Beispiele reicht, um die Kovarianz zu schaetzen. Guenstigste der drei Methoden.


In [1]:
import os
os.environ["N_PER_CLASS"] = "2048"
os.environ["SEED"] = "1"

In [2]:
from pathlib import Path

# ============================================================
# CONFIG
# LAPLACE
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO
    / "experiments"
    / "ag_news_qwen_lora"
    / "evaluate_saved_baseline.py"
)

RESULT_DIR = (
    MASTER_DIR
    / "method_results"
    / "laplace_qwen_agnews"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

# Standardwerte greifen beim interaktiven Ausfuehren wie bisher.
# Fuer SLURM-Batch-Jobs koennen SEED/N_PER_CLASS/N_SAMPLES per
# Umgebungsvariable ueberschrieben werden, ohne das Notebook zu duplizieren.
import os

SEED = int(os.environ.get("SEED", "1"))   # Zusatzlauf zur Multimodalitaets-Kontrolle (Seed 2 bleibt der Hauptlauf)
POSTERIOR_KEY_SEED = 2027   # gleich wie MILE/MFVI -- gleiches Subset

# ------------------------------------------------------------
# Posterior data (identisch zu 02_methods_v3.ipynb / 03_mfvi.ipynb)
# ------------------------------------------------------------

N_PER_CLASS = int(os.environ.get("N_PER_CLASS", "512"))        # erweitert von 128 -- 512 x 4 = 2048 examples
SEQ_LEN = 32

# ------------------------------------------------------------
# Prior (identisch zu MILE/MFVI -- geht als Prior-Praezision in
# die Laplace-Kovarianz ein)
# ------------------------------------------------------------

# War 1.0 (Standard-Default aus dem MILE-Paper). Bei 540672 Dimensionen
# ist das massiv zu breit: eine typische Stichprobe aus N(0, 1.0^2 * I)
# hat eine Norm von ~sqrt(540672)*1.0 ~= 735 -- weit weg von der
# tatsaechlichen MAP-Norm (~13.0). Das MILE-Paper selbst skaliert die
# Prior-Varianz fuer groessere Modelle runter (0.1-0.4 statt 1.0 fuer
# ihre CNN/ATT-Modelle, deutlich kleiner als unser 540k-dim LoRA-Raum).
# Empirisch hergeleitet aus der MAP-Norm: 13.0148 / sqrt(540672) ~= 0.0177.
PRIOR_STD = 0.02

# ------------------------------------------------------------
# Laplace
# ------------------------------------------------------------

N_SAMPLES = int(os.environ.get("N_SAMPLES", "10"))   # Posterior-Draws fuer die Auswertung (wie bei MILE/MFVI)

# Optionaler Skalierungsfaktor auf die geschaetzte Fisher-Diagonale
# (Standard-Trick aus "Laplace Redux", um Unter-/Ueberkonfidenz der
# rohen diagonalen Naeherung nachtraeglich zu korrigieren; 1.0 = kein
# Post-hoc-Tuning). Falls die Vorhersageunsicherheit spaeter zu klein
# oder zu gross wirkt, hier ansetzen.
FISHER_SCALE = 1.0

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64

# False (Default)  -> auswerten auf dem 128er-Posterior-Subset
#                      (schneller Dev-/Stabilitaets-Check).
# True              -> auswerten auf dem echten AG-News-Testset
#                      (7600 Beispiele, fuer die finalen Thesis-Zahlen).
EVAL_ON_TEST_SET = True   # direkt auf dem echten Testset -- N_PER_CLASS
                          # wurde konsistent mit MILE (adaptive) hochgesetzt

# ------------------------------------------------------------
# Automatic run name
# ------------------------------------------------------------

N_POSTERIOR_CONFIG = 4 * N_PER_CLASS

RUN_NAME = (
    f"laplace_seed{SEED}"
    f"_balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_fscale{FISHER_SCALE}"
    f"_samples{N_SAMPLES}"
)

print("======================================")
print("LAPLACE CONFIG")
print("======================================")
print("Run name:          ", RUN_NAME)
print("Seed:              ", SEED)
print("Examples/class:    ", N_PER_CLASS)
print("Total examples:    ", N_POSTERIOR_CONFIG)
print("Sequence length:   ", SEQ_LEN)
print("Prior std:         ", PRIOR_STD)
print("Fisher scale:      ", FISHER_SCALE)
print("Posterior samples: ", N_SAMPLES)
print("======================================")


LAPLACE CONFIG
Run name:           laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10
Seed:               1
Examples/class:     2048
Total examples:     8192
Sequence length:    32
Prior std:          0.02
Fisher scale:       1.0
Posterior samples:  10


In [3]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import copy
import json
import time
import runpy

import numpy as np
import jax
import jax.numpy as jnp

from jax import random


RESULT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

print("\nPaths:")
print("Qwen repo:", QWEN_REPO.exists())
print("Baseline script:", BASELINE_SCRIPT.exists())
print("Result directory:", RESULT_DIR)

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()

NOTEBOOK_START_TIME = time.time()  # Gesamtlaufzeit ab hier gemessen


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3
Devices: [CudaDevice(id=0)]

Paths:
Qwen repo: True
Baseline script: True
Result directory: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/laplace_qwen_agnews


In [4]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [5]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

test_metrics = baseline_objects["test_metrics"]

print("\nBaseline loaded")
print("Model:", type(env.s_model))

print("\nBaseline test metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {float(value):.6f}")


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246214
val_acc: 0.913667
val_ece: 0.009068
val_brier: 0.131597
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000

Baseline loaded
Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>

Baseline test metrics:
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000


In [6]:
# ============================================================
# SEED-SPEZIFISCHE PARAMS LADEN (nur noetig, wenn SEED != 2 -- die
# Baseline aus BASELINE_SCRIPT ist fest auf Seed 2 verdrahtet. Gleiche
# Technik wie in 02_mile_multichain.ipynb / 03_mfvi.ipynb / 05_deep_ensemble.ipynb:
# parametrisierte Kopie von evaluate_saved_baseline.py mit anderem
# CHECKPOINT_PATH/SEED.)
# ============================================================

CHECKPOINTS_ROOT = Path.home() / "Masterarbeit" / "checkpoints"


def load_seed_params(seed):
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    checkpoint_path = list(seed_dir.glob("*_pretrained_params.npy"))[0]

    eval_script_template = BASELINE_SCRIPT.read_text(encoding="utf-8")

    old_checkpoint_line = next(
        line for line in eval_script_template.splitlines()
        if line.strip().startswith("CHECKPOINT_PATH")
    )
    old_seed_line = next(
        line for line in eval_script_template.splitlines()
        if line.strip().startswith("SEED")
    )

    script_text = eval_script_template.replace(
        old_checkpoint_line, f'CHECKPOINT_PATH = r"{checkpoint_path}"',
    ).replace(
        old_seed_line, f"SEED = {seed}",
    )

    tmp_script_path = (
        QWEN_REPO / "experiments" / "ag_news_qwen_lora"
        / f"_tmp_eval_laplace_seed{seed}.py"
    )
    tmp_script_path.write_text(script_text, encoding="utf-8")
    try:
        seed_objects = runpy.run_path(str(tmp_script_path))
    finally:
        tmp_script_path.unlink(missing_ok=True)

    return seed_objects["params"]


if SEED != 2:
    print(f"Ersetze Baseline-Params (Seed 2) durch Seed {SEED} ...")
    params = load_seed_params(SEED)
    print(f"Params fuer Seed {SEED} geladen.")
else:
    print("SEED == 2 -> nutzt die bereits geladenen Baseline-Params.")


Ersetze Baseline-Params (Seed 2) durch Seed 1 ...


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed1/iiljya0g_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.247482
val_acc: 0.912250
val_ece: 0.008029
val_brier: 0.131916
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.252232
test_acc: 0.914474
test_ece: 0.007562
test_brier: 0.132536
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Params fuer Seed 1 geladen.


In [7]:
# ============================================================
# POSTERIOR SETUP (gleiches Subset, gleicher theta_map wie MILE/MFVI)
# ============================================================

posterior = qpu.setup_qwen_posterior(
    env=env,
    params=params,
    data=data,
    n_per_class=N_PER_CLASS,
    seq_len=SEQ_LEN,
    posterior_key_seed=POSTERIOR_KEY_SEED,
    prior_std=PRIOR_STD,
)

x_posterior = posterior.x_posterior
y_posterior = posterior.y_posterior
posterior_example_keys = posterior.posterior_example_keys
N_POSTERIOR_EXAMPLES = posterior.n_posterior_examples
subset_indices_host = posterior.subset_indices_host

theta_map = posterior.theta_map
rebuild_full_params = posterior.rebuild_full_params
qwen_single_example_nll = posterior.qwen_single_example_nll
qwen_log_posterior = posterior.qwen_log_posterior


Subset indices: [   0    1    4 ... 8128 8131 8135]
Labels: [0 0 0 ... 3 3 3]
Number of examples: 8192
input_ids: (8192, 32)
attention_mask: (8192, 32)
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True


### Diagonale empirische Fisher-Matrix

Fuer jedes der 128 Posterior-Beispiele wird der Gradient der Einzel-Beispiel-NLL an `theta_map` berechnet, quadriert und aufsummiert -- das ergibt die diagonale empirische Fisher-Information `F_diag` (Standard-Approximation der Hesse-Matrix fuer Laplace bei grossen Modellen, siehe Daxberger et al. 2021). Kein Optimierungsloop, nur ein `lax.scan` ueber die 128 Beispiele -- genauso teuer wie eine einzelne Gradientenauswertung von `qwen_log_posterior`.


In [8]:
# ============================================================
# DIAGONAL FISHER / GGN ESTIMATION AT MAP
# ============================================================


def per_example_nll(theta, x_single_batched, y_single_batched, key):
    candidate_params = rebuild_full_params(theta)
    return qwen_single_example_nll(
        candidate_params, x_single_batched, y_single_batched, key,
    )


per_example_grad_fn = jax.grad(per_example_nll, argnums=0)


def accumulate_fisher(fisher_diag, example_data):
    x_single, y_single, key = example_data

    x_single_batched = jax.tree.map(lambda array: array[None], x_single)
    y_single_batched = y_single[None]

    grad = per_example_grad_fn(
        theta_map, x_single_batched, y_single_batched, key,
    )

    fisher_diag = fisher_diag + jnp.square(grad)

    return fisher_diag, None


print("======================================")
print("COMPUTING DIAGONAL FISHER")
print("======================================")
print("Posterior examples:", N_POSTERIOR_EXAMPLES)

fisher_start_time = time.time()

initial_fisher = jnp.zeros_like(theta_map)

fisher_diag, _ = jax.lax.scan(
    accumulate_fisher,
    initial_fisher,
    (x_posterior, y_posterior, posterior_example_keys),
)

fisher_diag = jax.block_until_ready(fisher_diag)
fisher_duration = time.time() - fisher_start_time
fisher_diag_host = np.asarray(jax.device_get(fisher_diag))

print("Fisher diagonal finite:", np.isfinite(fisher_diag_host).all())
print("Fisher diagonal min/mean/max:", fisher_diag_host.min(), fisher_diag_host.mean(), fisher_diag_host.max())
print(f"Fisher-Berechnung Dauer: {fisher_duration:.1f}s ({fisher_duration/60:.1f} min)")

assert np.isfinite(fisher_diag_host).all(), "Diagonal Fisher contains NaN or Inf."


COMPUTING DIAGONAL FISHER
Posterior examples: 8192
Fisher diagonal finite: True
Fisher diagonal min/mean/max: 1.2592849e-11 7.8822603 7682.922
Fisher-Berechnung Dauer: 256.2s (4.3 min)


In [9]:
# ============================================================
# POSTERIOR COVARIANCE (Laplace)
# ============================================================

# posterior precision = (scaled) empirical Fisher + prior precision
prior_precision = 1.0 / (PRIOR_STD ** 2)

posterior_precision = FISHER_SCALE * fisher_diag + prior_precision
posterior_variance = 1.0 / posterior_precision
posterior_sigma = jnp.sqrt(posterior_variance)

posterior_sigma_host = np.asarray(jax.device_get(posterior_sigma))

print("Posterior sigma min/mean/max:", posterior_sigma_host.min(), posterior_sigma_host.mean(), posterior_sigma_host.max())
print("All finite:", np.isfinite(posterior_sigma_host).all())

assert np.isfinite(posterior_sigma_host).all(), "Laplace posterior sigma contains NaN or Inf."


Posterior sigma min/mean/max: 0.009909775 0.019972673 0.020000001
All finite: True


In [10]:
# ============================================================
# DRAW POSTERIOR SAMPLES FROM N(theta_map, diag(sigma^2))
# ============================================================

rng_key, sampling_key = random.split(rng_key)
sampling_keys = random.split(sampling_key, N_SAMPLES)


def draw_one_sample(key):
    epsilon = random.normal(key, theta_map.shape, dtype=theta_map.dtype)
    return theta_map + posterior_sigma * epsilon


sample_positions = jax.vmap(draw_one_sample)(sampling_keys)
sample_positions = jax.block_until_ready(sample_positions)

samples_host = np.asarray(jax.device_get(sample_positions))

finite_per_sample = np.isfinite(samples_host).all(axis=1)

print("Samples shape:", samples_host.shape)
print("All samples finite:", finite_per_sample.all())

assert finite_per_sample.all(), "At least one Laplace sample contains NaN or Inf."


Samples shape: (10, 540672)
All samples finite: True


In [11]:
# ============================================================
# BASIC SAMPLE DIAGNOSTICS
# ============================================================

distances_from_map = qpu.compute_distances_from_map(theta_map, sample_positions)


Distance summary:
Minimum: 14.6696005
Mean: 14.68527
Maximum: 14.697119


In [12]:
# ============================================================
# EVALUATION DATA (Posterior-Subset oder echtes Testset)
# ============================================================

if EVAL_ON_TEST_SET:
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set.")
else:
    x_eval, y_eval = x_posterior, y_posterior
    print("Evaluating on the posterior subset (dev/stability check).")

print("Evaluation examples:", int(y_eval.shape[0]))


Evaluating on the full AG News TEST set.
Evaluation examples: 7600


In [13]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

sample_probabilities, mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=samples_host,
        x_eval=x_eval,
        y_eval=y_eval,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)


Evaluation examples: 7600
Evaluation batch size: 64
Posterior samples: 10
Evaluating posterior sample 1/10
Evaluating posterior sample 2/10
Evaluating posterior sample 3/10
Evaluating posterior sample 4/10
Evaluating posterior sample 5/10
Evaluating posterior sample 6/10
Evaluating posterior sample 7/10
Evaluating posterior sample 8/10
Evaluating posterior sample 9/10
Evaluating posterior sample 10/10

Evaluation completed
Probability tensor shape: (10, 7600, 4)
Expected shape: (10, 7600, 4)
All probabilities finite: True


In [14]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)


Accuracy: 0.906973659992218
LPPD: -0.31146764755249023
Posterior predictive NLL: 0.31146764755249023
Brier Score: 0.15413545072078705
Mean predictive entropy: 0.5058245658874512
Mean expected entropy: 0.4490453600883484
Mean mutual information: 0.056779179722070694


In [15]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(mean_probabilities, y_eval, n_bins=15)
print("ECE:", float(ece))


ECE: 0.07829941064119339


In [16]:
# ============================================================
# LOG POSTERIOR VALUES FOR ALL SAMPLES
# ============================================================

sample_log_posteriors_host = qpu.compute_sample_log_posteriors(
    qwen_log_posterior,
    sample_positions,
)


Computed log posterior values: 10
Range: -486389.40625 to -483478.1875


In [17]:
# ============================================================
# SAVE RUN
# ============================================================

notebook_total_duration = time.time() - NOTEBOOK_START_TIME

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

summary.update({
    "run_name": RUN_NAME,
    "method": "Laplace",
    "covariance_structure": "diagonal",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "seed": SEED,
    "posterior_key_seed": POSTERIOR_KEY_SEED,

    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,

    "eval_on_test_set": EVAL_ON_TEST_SET,
    "n_eval_examples": int(y_eval.shape[0]),

    "prior_std": PRIOR_STD,
    "fisher_scale": FISHER_SCALE,
    "n_samples": N_SAMPLES,

    "fisher_diag_min": float(fisher_diag_host.min()),
    "fisher_diag_mean": float(fisher_diag_host.mean()),
    "fisher_diag_max": float(fisher_diag_host.max()),
    "fisher_duration_seconds": fisher_duration,

    "posterior_sigma_min": float(posterior_sigma_host.min()),
    "posterior_sigma_mean": float(posterior_sigma_host.mean()),
    "posterior_sigma_max": float(posterior_sigma_host.max()),

    "all_samples_finite": bool(np.isfinite(samples_host).all()),
    "n_parameter_dimensions": int(samples_host.shape[-1]),

    "minimum_distance_from_map": float(distances_from_map.min()),
    "mean_distance_from_map": float(distances_from_map.mean()),
    "maximum_distance_from_map": float(distances_from_map.max()),

    "minimum_log_posterior": float(sample_log_posteriors_host.min()),
    "mean_log_posterior": float(sample_log_posteriors_host.mean()),
    "maximum_log_posterior": float(sample_log_posteriors_host.max()),

    "notebook_total_duration_seconds": notebook_total_duration,
})

metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
    "fisher_diag": fisher_diag_host,
    "posterior_sigma": posterior_sigma_host,
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results:")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
print()
print(f"Fisher-Berechnung Dauer: {fisher_duration:.1f}s ({fisher_duration/60:.1f} min)")
print(f"Gesamtlaufzeit Notebook: {notebook_total_duration:.1f}s ({notebook_total_duration/60:.1f} min)")



RUN SAVED
Run: laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10
Samples:       /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/laplace_qwen_agnews/laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10_samples.npy
Probabilities: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/laplace_qwen_agnews/laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10_probabilities.npy
Metadata:      /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/laplace_qwen_agnews/laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10_metadata.npz
Summary:       /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/laplace_qwen_agnews/laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10_summary.json

Results:
Accuracy:            0.906973659992218
LPPD:                -0.31146764755249023
NLL:                 0.31146764755249023
Brier Score:         0.15413545072078705
ECE:                 0.07829941064119339
Mean pred. entropy:  0.505824625492096
Mean